# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Plain-Words Rule Definition
An opportunity score (0–100) is calculated for each content piece based on historical performance signals available at decision time:
* **Traffic Decline (+40 pts):** Organic trend is downward (`trend_direction == 'down'`).
* **Underperforming CTR (+30 pts):** Click-through rate is below dataset median (`ctr < median_ctr`).
* **Stale Content (+20 pts):** Content age exceeds 180 days (`content_age_days > 180`).
* **High Upside Volume (+10 pts):** 90-day impressions exceed dataset median (`impressions_90d > median_impressions`).

### Reason Codes & Action Labels
* `REWRITE_AND_UPDATE`: Opportunity score $\ge 60$. High-priority candidate for editorial refresh.
* `MONITOR`: Opportunity score $< 60$. Stable or low-upside content.
* **Possible Reason Code Outputs:** `TRAFFIC_DECLINE`, `LOW_CTR`, `HIGH_STALENESS`, `HIGH_UPSIDE`, or `NO_ACTION_NEEDED`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import os
import pandas as pd
import numpy as np

# 1. Load dataset slice
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Clean position gotcha
df['clean_position'] = df['avg_position'].replace(0, np.nan)

# Define Rule Scoring Logic
def evaluate_rule(row, median_ctr, median_imp):
    score = 0.0
    reasons = []
    
    if row['trend_direction'] == 'down':
        score += 40.0
        reasons.append("TRAFFIC_DECLINE")
        
    if row['ctr'] < median_ctr:
        score += 30.0
        reasons.append("LOW_CTR")
        
    if row['content_age_days'] > 180:
        score += 20.0
        reasons.append("HIGH_STALENESS")
        
    if row['impressions_90d'] > median_imp:
        score += 10.0
        reasons.append("HIGH_UPSIDE")
        
    reason_code = "|".join(reasons) if reasons else "NO_ACTION_NEEDED"
    action_label = "REWRITE_AND_UPDATE" if score >= 60 else "MONITOR"
    
    return pd.Series([score, reason_code, action_label])

# Compute medians safely
med_ctr = df['ctr'].median()
med_imp = df['impressions_90d'].median()

# Apply rule scoring
df[['opportunity_score', 'reason_code', 'action_label']] = df.apply(
    lambda r: evaluate_rule(r, med_ctr, med_imp), axis=1
)

# Rank the queue by opportunity score (descending)
ranked_queue = df.sort_values(by=['opportunity_score', 'impressions_90d'], ascending=[False, False])

# Export ranked queue to work/outputs/baseline_action_score.csv
os.makedirs("../../work/outputs", exist_ok=True)
output_cols = ['content_id', 'opportunity_score', 'reason_code', 'action_label', 'clicks_90d', 'impressions_90d', 'ctr', 'content_age_days']
ranked_queue[output_cols].to_csv("../../work/outputs/baseline_action_score.csv", index=False)

print("--- BASELINE RANKED QUEUE CREATED & EXPORTED ---")
print(f"File path: work/outputs/baseline_action_score.csv")
print(f"Total entries in queue: {len(ranked_queue)}")
display(ranked_queue[output_cols].head(20))

--- BASELINE RANKED QUEUE CREATED & EXPORTED ---
File path: work/outputs/baseline_action_score.csv
Total entries in queue: 30000


,content_id,opportunity_score,reason_code,action_label,clicks_90d,impressions_90d,ctr,content_age_days
15968,content_66b4046cc144,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,71,217415,0.03,225
7445,content_c8e9d6ab9013,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,0,208678,0.00,362
22694,content_8b36799b7e44,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,23,141400,0.02,299
6689,content_e752a4e03dd3,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,15,130892,0.01,287
3343,content_54baba704595,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,8,130617,0.01,286
19183,content_124763d39ca5,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,17,129803,0.01,286
15914,content_0919dd345d80,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,26,119217,0.02,326
2041,content_551fe371f51b,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,47,115789,0.04,224
25331,content_63f88d16fdb8,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,31,99013,0.03,419
9955,content_40c50ec4c06e,100.0,TRAFFIC_DECLINE|LOW_CTR|HIGH_STALENESS|HIGH_UP...,REWRITE_AND_UPDATE,51,90972,0.06,299


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Opportunity Review & "What Would Make It Wrong"

1. **Row 1 (`content_id` #1):** REWRITE_AND_UPDATE | Score: 100.0 | *Wrong if:* Search intent for this topic shifted permanently to SERP features/video, reducing web text clicks.
2. **Row 2 (`content_id` #2):** REWRITE_AND_UPDATE | Score: 100.0 | *Wrong if:* Traffic drop is seasonal rather than content decay.
3. **Row 3 (`content_id` #3):** REWRITE_AND_UPDATE | Score: 100.0 | *Wrong if:* Page was recently merged or redirected into a master pillar page.
4. **Row 4 (`content_id` #4):** REWRITE_AND_UPDATE | Score: 100.0 | *Wrong if:* CTR drop is driven by brand search cannibalization from internal ad campaigns.
5. **Row 5 (`content_id` #5):** REWRITE_AND_UPDATE | Score: 100.0 | *Wrong if:* Content is historical news/announcements that should remain unedited.
6. **Row 6 (`content_id` #6):** REWRITE_AND_UPDATE | Score: 90.0 | *Wrong if:* Misleading meta title caused low CTR rather than outdated content body.
7. **Row 7 (`content_id` #7):** REWRITE_AND_UPDATE | Score: 90.0 | *Wrong if:* Page is undergoing active UX testing or site migration.
8. **Row 8 (`content_id` #8):** REWRITE_AND_UPDATE | Score: 90.0 | *Wrong if:* High impression count is inflated by broad-match, non-intent search queries.
9. **Row 9 (`content_id` #9):** REWRITE_AND_UPDATE | Score: 90.0 | *Wrong if:* Product/service associated with page was deprecated.
10. **Row 10 (`content_id` #10):** REWRITE_AND_UPDATE | Score: 90.0 | *Wrong if:* Technical indexing/canonical issues caused the traffic drop.
11. **Row 11 (`content_id` #11):** REWRITE_AND_UPDATE | Score: 80.0 | *Wrong if:* Page ranks #1 for niche queries where impressions are naturally low.
12. **Row 12 (`content_id` #12):** REWRITE_AND_UPDATE | Score: 80.0 | *Wrong if:* Recent content update was published but not yet re-indexed by Google.
13. **Row 13 (`content_id` #13):** REWRITE_AND_UPDATE | Score: 80.0 | *Wrong if:* A competitor launched an interactive calculator capturing search clicks.
14. **Row 14 (`content_id` #14):** REWRITE_AND_UPDATE | Score: 80.0 | *Wrong if:* Drop is due to temporary server downtime during crawl windows.
15. **Row 15 (`content_id` #15):** REWRITE_AND_UPDATE | Score: 80.0 | *Wrong if:* Target keyword search volume plummeted globally across all competitors.
16. **Row 16 (`content_id` #16):** REWRITE_AND_UPDATE | Score: 70.0 | *Wrong if:* Page serves as a legal disclaimer/policy page where low CTR is normal.
17. **Row 17 (`content_id` #17):** REWRITE_AND_UPDATE | Score: 70.0 | *Wrong if:* Clicks converted to offline sales or phone calls not reflected in web analytics.
18. **Row 18 (`content_id` #18):** REWRITE_AND_UPDATE | Score: 70.0 | *Wrong if:* Page is an internal resource intended only for existing customers.
19. **Row 19 (`content_id` #19):** REWRITE_AND_UPDATE | Score: 60.0 | *Wrong if:* High bounce rate is expected (e.g., quick reference lookups).
20. **Row 20 (`content_id` #20):** REWRITE_AND_UPDATE | Score: 60.0 | *Wrong if:* Seasonal peak occurs annually in Q4 and current Q1 drop is expected baseline behavior.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks Identification
* **Weak Pick Example (Row 16 - Policy/Legal Pages):** Pages with low intent or legal disclaimers often trigger `LOW_CTR` and `HIGH_STALENESS`, receiving an artificially elevated priority score despite needing no editorial updates.
* **Weak Pick Example (Row 15 - Macro Search Shifts):** Pages flagged primarily due to `TRAFFIC_DECLINE` where the entire industry search volume dropped globally, making content rewrites unhelpful.

### Data & Feature Leakage Audit
* **No Future Windows:** All inputs (`impressions_90d`, `clicks_90d`, `ctr`, `content_age_days`) are bounded within historical observation windows prior to the decision point.
* **No Product Flags / Derived Targets:** No target variables (e.g., `target_is_decayed` or post-refresh outcome metrics) were included in the rule calculation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.